# Chapter 6.1: RadixAttention Advanced --- Cache-Aware Scheduling & Eviction Policies

## Learning Objectives

By the end of this notebook, you will:

1. **Understand cache-aware scheduling** and how SGLang prioritizes requests with existing cache hits
2. **Master eviction policies** including LRU, frequency-based, and size-aware strategies
3. **Explore cache warming strategies** for predictable workloads
4. **Analyze multi-level caching** architectures (GPU -> CPU -> disk)
5. **Understand cache coherence** challenges in distributed settings
6. **Perform performance analysis** of cache hit rate vs throughput
7. **Read and annotate** the SGLang eviction and scheduling source code
8. **Implement a custom eviction policy** as a hands-on exercise

## Prerequisites

- Understanding of radix trees and prefix caching (Part 5)
- Familiarity with LLM KV-cache fundamentals
- Basic understanding of GPU memory management

In [ ]:
import matplotlib.pyplot as pltimport matplotlib.patches as mpatchesimport numpy as np# Consistent color palette for all diagramsBLUE = '#4A90D9'GREEN = '#27AE60'ORANGE = '#F39C12'RED = '#E74C3C'PURPLE = '#8E44AD'LIGHT_BLUE = '#85C1E9'LIGHT_GREEN = '#82E0AA'LIGHT_ORANGE = '#F8C471'LIGHT_RED = '#F1948A'LIGHT_PURPLE = '#C39BD3'GRAY = '#95A5A6'DARK_GRAY = '#2C3E50'fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))fig.suptitle('Cache-Aware Scheduling: Impact on Hit Rate', fontsize=14,             fontweight='bold', color='#2C3E50')# Left: Without cache awareness (random order)ax1.set_xlim(0, 10)ax1.set_ylim(0, 10)ax1.axis('off')ax1.set_title('Without Cache Awareness (Random Order)', fontsize=11, fontweight='bold', color=RED, pad=10)requests_random = [    ("Req A (prefix=X)", RED, "MISS"),    ("Req B (prefix=Y)", ORANGE, "MISS"),    ("Req C (prefix=X)", RED, "evicted!"),    ("Req D (prefix=Z)", PURPLE, "MISS"),    ("Req E (prefix=Y)", ORANGE, "evicted!"),]for i, (label, color, status) in enumerate(requests_random):    y = 8 - i * 1.5    box = mpatches.FancyBboxPatch((0.5, y-0.35), 5, 0.7, boxstyle="round,pad=0.05",                                   facecolor=color, alpha=0.7, edgecolor='white', lw=1.5)    ax1.add_patch(box)    ax1.text(3, y, label, ha='center', va='center', fontsize=9, color='white', fontweight='bold')    ax1.text(7, y, status, ha='center', va='center', fontsize=10, color=RED, fontweight='bold')ax1.text(5, 0.5, 'Hit Rate: ~20%', ha='center', fontsize=12, fontweight='bold', color=RED,         bbox=dict(boxstyle='round,pad=0.3', facecolor=LIGHT_RED, alpha=0.3))# Right: With cache awareness (sorted by prefix)ax2.set_xlim(0, 10)ax2.set_ylim(0, 10)ax2.axis('off')ax2.set_title('With Cache Awareness (Sorted by Prefix)', fontsize=11, fontweight='bold', color=GREEN, pad=10)requests_sorted = [    ("Req A (prefix=X)", RED, "MISS"),    ("Req C (prefix=X)", RED, "HIT!"),    ("Req B (prefix=Y)", ORANGE, "MISS"),    ("Req E (prefix=Y)", ORANGE, "HIT!"),    ("Req D (prefix=Z)", PURPLE, "MISS"),]for i, (label, color, status) in enumerate(requests_sorted):    y = 8 - i * 1.5    box = mpatches.FancyBboxPatch((0.5, y-0.35), 5, 0.7, boxstyle="round,pad=0.05",                                   facecolor=color, alpha=0.7, edgecolor='white', lw=1.5)    ax2.add_patch(box)    ax2.text(3, y, label, ha='center', va='center', fontsize=9, color='white', fontweight='bold')    hit_color = GREEN if "HIT" in status else GRAY    ax2.text(7, y, status, ha='center', va='center', fontsize=10, color=hit_color, fontweight='bold')ax2.text(5, 0.5, 'Hit Rate: ~60%', ha='center', fontsize=12, fontweight='bold', color=GREEN,         bbox=dict(boxstyle='round,pad=0.3', facecolor=LIGHT_GREEN, alpha=0.3))plt.tight_layout(rect=[0, 0, 1, 0.93])plt.savefig("cache_aware_scheduling.png", dpi=150, bbox_inches='tight', facecolor='white')plt.show()

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part6/chapter_6.1_radix_advanced.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part6/chapter_6.1_radix_advanced.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Recap: RadixAttention and Prefix Caching

Before diving into advanced topics, let's briefly recall how RadixAttention works.

RadixAttention organizes KV-cache blocks in a **radix tree** structure where:
- Each node represents a sequence of token IDs
- Shared prefixes across requests map to the same tree path
- KV-cache blocks are attached to tree nodes and reused across requests

```
              [ROOT]
             /      \
     [system_prompt]  [other_prefix]
       /      \
  [user_A]  [user_B]     <-- Different continuations share system prompt cache
```

The key insight: when a new request arrives that shares a prefix with a previous request, we can **skip recomputation** of that prefix's KV-cache entirely.

## 2. Cache-Aware Scheduling

### 2.1 The Core Problem

Standard LLM serving schedulers pick requests in FIFO order or by priority. But this ignores a crucial optimization opportunity: **some requests can reuse existing cache entries while others cannot.**

Consider this scenario:
- Request A: shares 90% prefix with cached data (only 10% new tokens to compute)
- Request B: shares 0% prefix (100% new computation needed)

A cache-aware scheduler would prioritize Request A because:
1. It requires fewer prefill FLOPs
2. It can start generating tokens sooner
3. It improves overall throughput by reducing GPU idle time

### 2.2 Architecture Overview

```
                    Incoming Requests
                          |
                          v
              +------------------------+
              | Cache-Aware Scheduler  |
              |                        |
              |  1. Check radix tree   |
              |     for prefix match   |
              |  2. Compute cache hit  |
              |     ratio for each req |
              |  3. Sort by priority:  |
              |     hit_ratio * weight |
              |     + urgency_score    |
              +------------------------+
                          |
                   Scheduled Batch
                          |
                          v
              +------------------------+
              |    GPU Execution       |
              |  (prefill + decode)    |
              +------------------------+
```

### 2.3 Source Code Analysis: Cache-Aware Scheduling in SGLang

The cache-aware scheduling logic lives primarily in `python/sglang/srt/managers/scheduler.py`. Let's trace the key functions.

**Key function: `get_new_batch_prefill`**

```python
# From sglang/srt/managers/scheduler.py

def get_new_batch_prefill(self):
    """
    Select requests for the next prefill batch.
    Cache-aware: requests with longer prefix matches are prioritized.
    """
    # Step 1: For each waiting request, compute prefix match length
    for req in self.waiting_queue:
        req.prefix_len = self.tree_cache.match_prefix(req.input_ids)
    
    # Step 2: Sort by prefix match ratio (descending)
    # Requests with more cache hits come first
    if self.schedule_policy == "lpm":  # Longest Prefix Match
        self.waiting_queue.sort(
            key=lambda r: -r.prefix_len  # Higher prefix = higher priority
        )
    
    # Step 3: Build batch respecting memory constraints
    new_batch = []
    available_slots = self.max_running_requests - len(self.running_batch)
    
    for req in self.waiting_queue:
        if len(new_batch) >= available_slots:
            break
        # Only the non-cached portion needs new memory
        new_tokens = len(req.input_ids) - req.prefix_len
        if self.can_allocate(new_tokens + req.max_new_tokens):
            new_batch.append(req)
    
    return new_batch
```

**Key insight:** The `match_prefix` call walks the radix tree to find the longest cached prefix. Requests with longer matches need fewer new KV-cache blocks, making them cheaper to prefill.

In [ ]:
# Simulation: Cache-Aware Scheduling vs FIFO
import random
import time
from typing import List, Tuple

class MockRequest:
    """Simulated request with prefix cache information."""
    def __init__(self, req_id: int, total_tokens: int, cached_prefix_len: int):
        self.req_id = req_id
        self.total_tokens = total_tokens
        self.cached_prefix_len = cached_prefix_len
        self.new_tokens_needed = total_tokens - cached_prefix_len
        self.cache_hit_ratio = cached_prefix_len / total_tokens if total_tokens > 0 else 0
    
    def __repr__(self):
        return (f"Req({self.req_id}: total={self.total_tokens}, "
                f"cached={self.cached_prefix_len}, "
                f"new={self.new_tokens_needed}, "
                f"hit={self.cache_hit_ratio:.1%})")

# Generate a batch of requests with varying cache hit ratios
random.seed(42)
requests = []
for i in range(10):
    total = random.randint(100, 500)
    # Some requests have high cache hits, others low
    cached = random.randint(0, total)
    requests.append(MockRequest(i, total, cached))

print("=== Incoming Requests ===")
for r in requests:
    print(f"  {r}")

In [ ]:
def fifo_schedule(requests: List[MockRequest], max_batch_tokens: int) -> List[MockRequest]:
    """FIFO scheduling: process requests in arrival order."""
    batch = []
    total_new_tokens = 0
    for req in requests:
        if total_new_tokens + req.new_tokens_needed <= max_batch_tokens:
            batch.append(req)
            total_new_tokens += req.new_tokens_needed
    return batch

def cache_aware_schedule(requests: List[MockRequest], max_batch_tokens: int) -> List[MockRequest]:
    """Cache-aware scheduling: prioritize high cache-hit requests."""
    # Sort by cache hit ratio (descending) -- longest prefix match first
    sorted_reqs = sorted(requests, key=lambda r: -r.cache_hit_ratio)
    batch = []
    total_new_tokens = 0
    for req in sorted_reqs:
        if total_new_tokens + req.new_tokens_needed <= max_batch_tokens:
            batch.append(req)
            total_new_tokens += req.new_tokens_needed
    return batch

MAX_BATCH_TOKENS = 600

fifo_batch = fifo_schedule(requests, MAX_BATCH_TOKENS)
cache_batch = cache_aware_schedule(requests, MAX_BATCH_TOKENS)

print(f"\n=== FIFO Scheduling (max {MAX_BATCH_TOKENS} new tokens) ===")
print(f"Batch size: {len(fifo_batch)} requests")
total_served = sum(r.total_tokens for r in fifo_batch)
total_new = sum(r.new_tokens_needed for r in fifo_batch)
print(f"Total tokens served: {total_served}, New tokens computed: {total_new}")
print(f"Effective throughput ratio: {total_served / total_new:.2f}x")

print(f"\n=== Cache-Aware Scheduling (max {MAX_BATCH_TOKENS} new tokens) ===")
print(f"Batch size: {len(cache_batch)} requests")
total_served = sum(r.total_tokens for r in cache_batch)
total_new = sum(r.new_tokens_needed for r in cache_batch)
print(f"Total tokens served: {total_served}, New tokens computed: {total_new}")
print(f"Effective throughput ratio: {total_served / total_new:.2f}x")

### 2.4 Scheduling Policy Options in SGLang

SGLang supports multiple scheduling policies configured via `--schedule-policy`:

| Policy | Description | Best For |
|--------|-------------|----------|
| `lpm` | Longest Prefix Match - prioritize requests with longest cached prefix | Chatbots with system prompts |
| `fcfs` | First Come First Served - standard FIFO | Fairness-critical workloads |
| `random` | Random selection | Baseline comparison |
| `dfs-weight` | Depth-first search weighted | Tree-structured generation |

The `lpm` policy is the default and delivers the best throughput for most workloads because it maximizes KV-cache reuse.

## 3. Eviction Policies

### 3.1 Why Eviction Matters

GPU memory is finite. When the radix tree grows too large, we must evict some cached KV blocks. The eviction policy determines **which blocks to remove** and directly impacts:

- **Cache hit rate**: Good eviction keeps frequently-reused prefixes
- **Throughput**: Higher cache hit rate = less redundant computation
- **Latency**: Evicting the wrong blocks forces re-computation

### 3.2 LRU (Least Recently Used) Eviction

The default and most common policy. Each node in the radix tree tracks when it was last accessed.

```
Radix Tree with LRU timestamps:

          [ROOT]
         /      \
   [A: t=100]  [B: t=50]    <-- B is older, evict first
     /    \
[C: t=90] [D: t=20]         <-- D is oldest leaf, evict first
```

**Algorithm:**
1. Maintain a sorted structure of leaf nodes by last-access time
2. When memory is needed, evict the least-recently-used leaf
3. If a node has no children after eviction, it becomes a new leaf candidate

### 3.3 Source Code: LRU Eviction in RadixCache

```python
# From sglang/srt/mem_cache/radix_cache.py

class RadixCache:
    def __init__(self, ...):
        self.root_node = TreeNode()  # Root of the radix tree
        self.evictable_size_ = 0     # Total evictable blocks
    
    def evict(self, num_tokens: int):
        """
        Evict enough cache entries to free `num_tokens` worth of KV blocks.
        Uses LRU: evict leaves with the oldest last_access_time first.
        """
        if self.disable:
            return
        
        # Collect all evictable leaves (nodes with ref_count == 0)
        leaves = self._collect_leaves()
        
        # Sort by last_access_time (ascending = oldest first)
        heapq.heapify(leaves)  # Min-heap by last_access_time
        
        evicted = 0
        while evicted < num_tokens and leaves:
            node = heapq.heappop(leaves)
            
            if node.ref_count > 0:
                continue  # Skip nodes still in use
            
            # Free the KV-cache blocks for this node
            num_blocks = len(node.value)  # value = list of block indices
            self.token_to_kv_pool.free(node.value)
            evicted += num_blocks * self.block_size
            
            # Remove node from tree
            parent = node.parent
            del parent.children[node.key]
            
            # Parent may become a new leaf
            if len(parent.children) == 0 and parent.ref_count == 0:
                heapq.heappush(leaves, parent)
        
        self.evictable_size_ -= evicted
        return evicted
```

**Key details:**
- `ref_count`: tracks how many active requests reference this node. Only nodes with `ref_count == 0` can be evicted.
- `value`: stores the actual KV-cache block indices in the token pool
- Eviction cascades upward: when a leaf is evicted, its parent may become evictable

In [ ]:
import heapq
from collections import defaultdict

class TreeNode:
    """Simplified radix tree node for eviction simulation."""
    _counter = 0  # For tie-breaking in heap
    
    def __init__(self, key="", num_blocks=0, parent=None):
        self.key = key
        self.num_blocks = num_blocks  # Number of KV-cache blocks
        self.parent = parent
        self.children = {}
        self.ref_count = 0  # Active references
        self.last_access_time = 0
        self.access_count = 0  # For frequency-based eviction
        TreeNode._counter += 1
        self._id = TreeNode._counter
    
    def __lt__(self, other):
        """For heap ordering: compare by last_access_time."""
        if self.last_access_time == other.last_access_time:
            return self._id < other._id
        return self.last_access_time < other.last_access_time
    
    def is_leaf(self):
        return len(self.children) == 0
    
    def __repr__(self):
        return f"Node('{self.key}', blocks={self.num_blocks}, t={self.last_access_time}, refs={self.ref_count})"


class SimpleRadixCache:
    """Simplified radix cache with pluggable eviction policies."""
    
    def __init__(self, max_blocks: int):
        self.root = TreeNode("root")
        self.max_blocks = max_blocks
        self.used_blocks = 0
        self.time = 0
        self.stats = {"hits": 0, "misses": 0, "evictions": 0}
    
    def insert(self, path: List[str], blocks_per_node: int = 4) -> int:
        """Insert a path into the cache. Returns number of cache hits."""
        self.time += 1
        node = self.root
        hits = 0
        
        for key in path:
            if key in node.children:
                # Cache hit for this segment
                node = node.children[key]
                node.last_access_time = self.time
                node.access_count += 1
                hits += 1
            else:
                # Cache miss: need to allocate new blocks
                while self.used_blocks + blocks_per_node > self.max_blocks:
                    evicted = self._evict_lru()
                    if not evicted:
                        break  # Cannot evict anything
                
                new_node = TreeNode(key, blocks_per_node, parent=node)
                new_node.last_access_time = self.time
                new_node.access_count = 1
                node.children[key] = new_node
                self.used_blocks += blocks_per_node
                node = new_node
        
        self.stats["hits"] += hits
        self.stats["misses"] += len(path) - hits
        return hits
    
    def _collect_leaves(self, node=None):
        """Collect all evictable leaf nodes."""
        if node is None:
            node = self.root
        leaves = []
        if node.is_leaf() and node != self.root and node.ref_count == 0:
            leaves.append(node)
        for child in node.children.values():
            leaves.extend(self._collect_leaves(child))
        return leaves
    
    def _evict_lru(self) -> bool:
        """Evict the least recently used leaf node."""
        leaves = self._collect_leaves()
        if not leaves:
            return False
        
        # Find the LRU leaf
        victim = min(leaves, key=lambda n: n.last_access_time)
        
        # Remove from parent
        if victim.parent and victim.key in victim.parent.children:
            del victim.parent.children[victim.key]
        
        self.used_blocks -= victim.num_blocks
        self.stats["evictions"] += 1
        return True
    
    def get_hit_rate(self) -> float:
        total = self.stats["hits"] + self.stats["misses"]
        return self.stats["hits"] / total if total > 0 else 0
    
    def print_tree(self, node=None, indent=0):
        if node is None:
            node = self.root
        print(" " * indent + str(node))
        for child in node.children.values():
            self.print_tree(child, indent + 2)

print("SimpleRadixCache class defined successfully.")

In [ ]:
# Demo: LRU Eviction in Action
cache = SimpleRadixCache(max_blocks=20)  # Small cache for demonstration

# Simulate requests with shared prefixes
requests_sequence = [
    ["system", "user_a", "response_a1"],  # First request
    ["system", "user_a", "response_a2"],  # Same user, different response
    ["system", "user_b", "response_b1"],  # Different user, same system prompt
    ["system", "user_c", "response_c1"],  # Another user
    ["system", "user_a", "response_a3"],  # Back to user A (should hit cache)
    ["other_sys", "user_d", "response_d1"],  # Completely different prefix
]

print("=== Simulating Request Sequence ===")
for i, req_path in enumerate(requests_sequence):
    hits = cache.insert(req_path)
    print(f"\nRequest {i}: {' -> '.join(req_path)}")
    print(f"  Cache hits: {hits}/{len(req_path)} segments")
    print(f"  Used blocks: {cache.used_blocks}/{cache.max_blocks}")
    print(f"  Evictions so far: {cache.stats['evictions']}")

print(f"\n=== Final Stats ===")
print(f"Overall cache hit rate: {cache.get_hit_rate():.1%}")
print(f"\nTree structure:")
cache.print_tree()

### 3.4 Frequency-Based Eviction

LRU can be suboptimal when some prefixes are accessed frequently but with gaps between accesses. A **frequency-based** policy (similar to LFU - Least Frequently Used) considers how many times a prefix has been accessed.

**Trade-offs:**

| Policy | Pros | Cons |
|--------|------|------|
| LRU | Simple, good for temporal locality | Can evict frequently-used items |
| LFU | Keeps popular items | Stale popular items never evict |
| LRU + Frequency Hybrid | Best of both worlds | More complex bookkeeping |

### 3.5 Size-Aware Eviction

Not all cache entries are equal in size. A size-aware policy considers the **cost-benefit ratio**:

```
eviction_score = last_access_time / num_blocks
```

Small nodes that were recently accessed should be kept (high score). Large nodes that haven't been accessed recently should be evicted first (low score).

In [ ]:
# Implementing different eviction policies for comparison

class EvictionPolicy:
    """Base class for eviction policies."""
    def select_victim(self, leaves: List[TreeNode]) -> TreeNode:
        raise NotImplementedError

class LRUEviction(EvictionPolicy):
    """Least Recently Used."""
    def select_victim(self, leaves):
        return min(leaves, key=lambda n: n.last_access_time)

class LFUEviction(EvictionPolicy):
    """Least Frequently Used."""
    def select_victim(self, leaves):
        return min(leaves, key=lambda n: n.access_count)

class SizeAwareEviction(EvictionPolicy):
    """Evict entries with lowest value (access_time * frequency / size)."""
    def select_victim(self, leaves):
        def score(node):
            # Higher score = more valuable (keep it)
            return (node.last_access_time * node.access_count) / max(node.num_blocks, 1)
        return min(leaves, key=score)

class HybridEviction(EvictionPolicy):
    """Weighted combination of recency and frequency."""
    def __init__(self, recency_weight=0.7, frequency_weight=0.3):
        self.rw = recency_weight
        self.fw = frequency_weight
    
    def select_victim(self, leaves):
        max_time = max(n.last_access_time for n in leaves)
        max_freq = max(n.access_count for n in leaves)
        
        def score(node):
            recency_score = node.last_access_time / max_time if max_time > 0 else 0
            freq_score = node.access_count / max_freq if max_freq > 0 else 0
            return self.rw * recency_score + self.fw * freq_score
        
        return min(leaves, key=score)

print("Eviction policies defined:")
print("  - LRUEviction")
print("  - LFUEviction")
print("  - SizeAwareEviction")
print("  - HybridEviction")

In [ ]:
# Benchmarking different eviction policies
import random

class ConfigurableCache(SimpleRadixCache):
    """Cache with pluggable eviction policy."""
    def __init__(self, max_blocks, policy: EvictionPolicy):
        super().__init__(max_blocks)
        self.policy = policy
    
    def _evict_lru(self) -> bool:
        leaves = self._collect_leaves()
        if not leaves:
            return False
        victim = self.policy.select_victim(leaves)
        if victim.parent and victim.key in victim.parent.children:
            del victim.parent.children[victim.key]
        self.used_blocks -= victim.num_blocks
        self.stats["evictions"] += 1
        return True


def generate_workload(n_requests=200, n_users=5, n_topics=3):
    """Generate a realistic workload with shared prefixes."""
    workload = []
    system_prompts = [f"sys_{i}" for i in range(n_topics)]
    users = [f"user_{i}" for i in range(n_users)]
    
    for _ in range(n_requests):
        # Zipf-like distribution: some users/topics are more popular
        sys = random.choices(system_prompts, weights=[5, 3, 1])[0]
        user = random.choices(users, weights=[5, 3, 2, 1, 1])[0]
        response = f"resp_{random.randint(0, 1000)}"
        workload.append([sys, user, response])
    
    return workload

# Run benchmark
random.seed(42)
workload = generate_workload(300)

policies = {
    "LRU": LRUEviction(),
    "LFU": LFUEviction(),
    "Size-Aware": SizeAwareEviction(),
    "Hybrid (0.7R+0.3F)": HybridEviction(0.7, 0.3),
}

print("=== Eviction Policy Benchmark ===")
print(f"Workload: {len(workload)} requests, cache size: 24 blocks\n")

results = {}
for name, policy in policies.items():
    cache = ConfigurableCache(max_blocks=24, policy=policy)
    for path in workload:
        cache.insert(path)
    
    hit_rate = cache.get_hit_rate()
    results[name] = hit_rate
    print(f"{name:25s}: hit_rate={hit_rate:.1%}, evictions={cache.stats['evictions']}")

In [ ]:
# Visualize eviction policy comparison
try:
    import matplotlib.pyplot as plt
    
    fig, ax = plt.subplots(figsize=(10, 5))
    policies_names = list(results.keys())
    hit_rates = [results[p] * 100 for p in policies_names]
    
    bars = ax.bar(policies_names, hit_rates, color=['#2196F3', '#4CAF50', '#FF9800', '#9C27B0'])
    ax.set_ylabel('Cache Hit Rate (%)')
    ax.set_title('Eviction Policy Comparison')
    ax.set_ylim(0, 100)
    
    for bar, rate in zip(bars, hit_rates):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                f'{rate:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not available. Textual results above.")

## 4. Cache Warming Strategies

### 4.1 Problem: Cold Start

When the server starts, the cache is empty. The first batch of requests will all miss, leading to:
- Higher latency for initial requests
- Suboptimal throughput during warmup period
- Poor user experience for the first N requests

### 4.2 Warming Strategies

**Strategy 1: Pre-populate System Prompts**
```python
# At server startup, run prefill for known system prompts
system_prompts = load_common_prompts()  # e.g., from config
for prompt in system_prompts:
    tokenized = tokenizer.encode(prompt)
    # Run prefill and store KV-cache in radix tree
    scheduler.warm_prefix(tokenized)
```

**Strategy 2: Replay Recent History**
```python
# Load access patterns from previous server session
recent_prefixes = load_access_log(last_n_hours=1)
for prefix_tokens in recent_prefixes:
    scheduler.warm_prefix(prefix_tokens)
```

**Strategy 3: Predictive Warming**
```python
# Use access patterns to predict next likely prefixes
# e.g., time-of-day patterns, trending topics
predicted = predict_next_prefixes(current_time)
for prefix_tokens in predicted:
    scheduler.warm_prefix(prefix_tokens)
```

### 4.3 Trade-offs

| Strategy | Latency Impact | Memory Cost | Complexity |
|----------|---------------|-------------|------------|
| No warming | High initial latency | None | None |
| System prompt pre-pop | Low | Moderate | Low |
| History replay | Very low | High | Medium |
| Predictive warming | Lowest (if accurate) | Variable | High |

In [ ]:
# Simulate cold start vs warm start

def simulate_with_warming(workload, max_blocks, warm_prefixes=None):
    """Simulate cache performance with optional warming."""
    cache = ConfigurableCache(max_blocks=max_blocks, policy=LRUEviction())
    
    # Warm the cache if prefixes provided
    if warm_prefixes:
        for prefix in warm_prefixes:
            cache.insert(prefix)
        # Reset stats after warming (warming misses don't count)
        cache.stats = {"hits": 0, "misses": 0, "evictions": 0}
    
    # Process workload
    batch_hit_rates = []
    batch_size = 20
    for i in range(0, len(workload), batch_size):
        batch = workload[i:i+batch_size]
        batch_hits = sum(cache.insert(path) for path in batch)
        batch_total = sum(len(path) for path in batch)
        batch_hit_rates.append(batch_hits / batch_total if batch_total > 0 else 0)
    
    return batch_hit_rates, cache.get_hit_rate()

random.seed(42)
workload = generate_workload(200)

# Cold start
cold_rates, cold_overall = simulate_with_warming(workload, max_blocks=30)

# Warm start: pre-populate system prompts
warm_prefixes = [["sys_0"], ["sys_1"], ["sys_2"]]  # Just system prompt prefixes
warm_rates, warm_overall = simulate_with_warming(workload, max_blocks=30, warm_prefixes=warm_prefixes)

print(f"Cold start overall hit rate: {cold_overall:.1%}")
print(f"Warm start overall hit rate: {warm_overall:.1%}")
print(f"\nBatch-by-batch hit rates (first 5 batches):")
print(f"  Cold: {[f'{r:.1%}' for r in cold_rates[:5]]}")
print(f"  Warm: {[f'{r:.1%}' for r in warm_rates[:5]]}")

## 5. Multi-Level Caching: GPU -> CPU -> Disk

> **Note:** Multi-level KV-cache caching (GPU->CPU->disk) is a research direction and design exploration. SGLang's current implementation primarily uses GPU-level caching with LRU eviction. The architectures described here represent potential extensions.

### 5.1 The Memory Hierarchy Problem

GPU HBM is expensive and limited (e.g., 80GB on A100). For large-scale deployments:
- **GPU HBM**: Fast (2 TB/s), small (40-80 GB)
- **CPU DRAM**: Medium speed (100 GB/s), larger (256-512 GB)
- **NVMe SSD**: Slower (7 GB/s), very large (TB+)

A multi-level cache uses all three tiers:

```
                    Request
                       |
                       v
              +------------------+
              |  GPU HBM Cache   |  <-- Fastest, smallest
              |  (Active cache)  |      Hit = instant reuse
              +--------+---------+
                       | miss
                       v
              +------------------+
              |  CPU DRAM Cache  |  <-- Medium speed
              |  (Evicted from   |      Hit = GPU upload needed
              |   GPU recently)  |      (~1ms for 1MB)
              +--------+---------+
                       | miss
                       v
              +------------------+
              |   Disk Cache     |  <-- Slowest, largest
              |  (Persistent     |      Hit = disk read + upload
              |   across restarts)|     (~10ms for 1MB)
              +------------------+
                       | miss
                       v
                 Recompute from scratch
```

### 5.2 Transfer Costs

| Transfer | Bandwidth | Latency for 1MB KV block |
|----------|-----------|-------------------------|
| GPU HBM (cache hit) | 2 TB/s | ~0.5 us |
| CPU -> GPU (PCIe 4.0) | 32 GB/s | ~31 us |
| CPU -> GPU (NVLink) | 600 GB/s | ~1.7 us |
| Disk -> CPU -> GPU | ~7 GB/s | ~143 us |
| Recompute (prefill) | N/A | ~1-10 ms |

**Key insight:** Even disk access (~0.14 ms) is much faster than recomputation (~1-10 ms), making multi-level caching worthwhile.

In [ ]:
# Multi-Level Cache Simulation

class MultiLevelCache:
    """Simulated 3-tier cache: GPU -> CPU -> Disk."""
    
    def __init__(self, gpu_capacity, cpu_capacity, disk_capacity):
        self.gpu = {}   # key -> (value, access_time)
        self.cpu = {}   # key -> (value, access_time)
        self.disk = {}  # key -> (value, access_time)
        
        self.gpu_capacity = gpu_capacity
        self.cpu_capacity = cpu_capacity
        self.disk_capacity = disk_capacity
        
        self.time = 0
        self.stats = {
            "gpu_hits": 0, "cpu_hits": 0, "disk_hits": 0, "misses": 0,
            "total_latency_us": 0
        }
        
        # Latency in microseconds
        self.GPU_HIT_LATENCY = 0.5
        self.CPU_TO_GPU_LATENCY = 31
        self.DISK_TO_GPU_LATENCY = 143
        self.RECOMPUTE_LATENCY = 5000  # 5ms average prefill
    
    def access(self, key: str) -> float:
        """Access a cache entry. Returns latency in microseconds."""
        self.time += 1
        
        # Check GPU first
        if key in self.gpu:
            self.gpu[key] = (self.gpu[key][0], self.time)
            self.stats["gpu_hits"] += 1
            self.stats["total_latency_us"] += self.GPU_HIT_LATENCY
            return self.GPU_HIT_LATENCY
        
        # Check CPU
        if key in self.cpu:
            value = self.cpu.pop(key)
            self._gpu_insert(key, value[0])  # Promote to GPU
            self.stats["cpu_hits"] += 1
            self.stats["total_latency_us"] += self.CPU_TO_GPU_LATENCY
            return self.CPU_TO_GPU_LATENCY
        
        # Check disk
        if key in self.disk:
            value = self.disk.pop(key)
            self._gpu_insert(key, value[0])  # Promote to GPU
            self.stats["disk_hits"] += 1
            self.stats["total_latency_us"] += self.DISK_TO_GPU_LATENCY
            return self.DISK_TO_GPU_LATENCY
        
        # Miss: compute and insert
        self._gpu_insert(key, f"kv_data_{key}")
        self.stats["misses"] += 1
        self.stats["total_latency_us"] += self.RECOMPUTE_LATENCY
        return self.RECOMPUTE_LATENCY
    
    def _gpu_insert(self, key, value):
        """Insert into GPU cache, evicting to CPU if needed."""
        if len(self.gpu) >= self.gpu_capacity:
            # Evict LRU from GPU to CPU
            lru_key = min(self.gpu, key=lambda k: self.gpu[k][1])
            evicted = self.gpu.pop(lru_key)
            self._cpu_insert(lru_key, evicted[0])
        self.gpu[key] = (value, self.time)
    
    def _cpu_insert(self, key, value):
        """Insert into CPU cache, evicting to disk if needed."""
        if len(self.cpu) >= self.cpu_capacity:
            lru_key = min(self.cpu, key=lambda k: self.cpu[k][1])
            evicted = self.cpu.pop(lru_key)
            self._disk_insert(lru_key, evicted[0])
        self.cpu[key] = (value, self.time)
    
    def _disk_insert(self, key, value):
        """Insert into disk cache, evicting LRU if needed."""
        if len(self.disk) >= self.disk_capacity:
            lru_key = min(self.disk, key=lambda k: self.disk[k][1])
            del self.disk[lru_key]  # Evict permanently
        self.disk[key] = (value, self.time)
    
    def print_stats(self):
        total = sum([
            self.stats["gpu_hits"], self.stats["cpu_hits"],
            self.stats["disk_hits"], self.stats["misses"]
        ])
        print(f"Total accesses:   {total}")
        print(f"  GPU hits:       {self.stats['gpu_hits']} ({self.stats['gpu_hits']/total:.1%})")
        print(f"  CPU hits:       {self.stats['cpu_hits']} ({self.stats['cpu_hits']/total:.1%})")
        print(f"  Disk hits:      {self.stats['disk_hits']} ({self.stats['disk_hits']/total:.1%})")
        print(f"  Misses:         {self.stats['misses']} ({self.stats['misses']/total:.1%})")
        avg_latency = self.stats['total_latency_us'] / total
        print(f"  Avg latency:    {avg_latency:.1f} us")

# Compare single-level vs multi-level
random.seed(42)

# Generate access pattern: Zipf distribution (some prefixes very popular)
n_unique_prefixes = 50
n_accesses = 500
# Zipf weights
weights = [1.0 / (i + 1) for i in range(n_unique_prefixes)]
access_sequence = random.choices(
    [f"prefix_{i}" for i in range(n_unique_prefixes)],
    weights=weights,
    k=n_accesses
)

print("=== Single-Level Cache (GPU only, capacity=10) ===")
single = MultiLevelCache(gpu_capacity=10, cpu_capacity=0, disk_capacity=0)
for key in access_sequence:
    single.access(key)
single.print_stats()

print("\n=== Multi-Level Cache (GPU=10, CPU=20, Disk=50) ===")
multi = MultiLevelCache(gpu_capacity=10, cpu_capacity=20, disk_capacity=50)
for key in access_sequence:
    multi.access(key)
multi.print_stats()

## 6. Cache Coherence in Distributed Settings

### 6.1 The Distributed Cache Challenge

When running SGLang with multiple workers (e.g., data parallelism), each worker has its own radix tree and KV-cache. This creates coherence challenges:

```
                   Load Balancer
                  /      |      \
          Worker 0    Worker 1    Worker 2
          Cache A     Cache B     Cache C
          
Problem: A request with prefix "sys_prompt + user_A" might be:
  - Cached on Worker 0 (from a previous request)
  - NOT cached on Worker 1 or 2
  
If the load balancer sends it to Worker 1, we lose the cache benefit!
```

### 6.2 Solutions

**Solution 1: Cache-Aware Routing**
```python
# Hash-based routing: same prefix -> same worker
def route_request(request, n_workers):
    prefix_hash = hash(tuple(request.input_ids[:SYSTEM_PROMPT_LEN]))
    return prefix_hash % n_workers
```

**Solution 2: Cache State Broadcasting**
```python
# Each worker periodically broadcasts its cache state
# Router picks the worker with the best cache match
def route_with_cache_state(request, workers):
    best_worker = max(
        workers,
        key=lambda w: w.cache.match_prefix_length(request.input_ids)
    )
    return best_worker
```

**Solution 3: Shared Cache Index (SGLang DataParallelController approach)**
- Maintain a global index of which worker caches which prefixes
- Route requests to the worker with the best match
- Used by SGLang's `DataParallelController`

In [ ]:
# Simulate distributed cache with different routing strategies

class DistributedCache:
    """Simulates multiple workers with independent caches."""
    
    def __init__(self, n_workers, cache_size_per_worker):
        self.workers = [
            SimpleRadixCache(max_blocks=cache_size_per_worker) 
            for _ in range(n_workers)
        ]
        self.n_workers = n_workers
    
    def round_robin_route(self, request_idx):
        """Simple round-robin routing (cache-unaware)."""
        return request_idx % self.n_workers
    
    def hash_route(self, path):
        """Route based on prefix hash (cache-aware)."""
        # Hash the first element (system prompt) for consistent routing
        return hash(path[0]) % self.n_workers
    
    def best_match_route(self, path):
        """Route to worker with best cache match (ideal but expensive)."""
        best_worker = 0
        best_hits = -1
        for i, worker in enumerate(self.workers):
            # Check how many segments match in this worker's cache
            node = worker.root
            hits = 0
            for key in path:
                if key in node.children:
                    hits += 1
                    node = node.children[key]
                else:
                    break
            if hits > best_hits:
                best_hits = hits
                best_worker = i
        return best_worker
    
    def process_workload(self, workload, routing_strategy="round_robin"):
        """Process workload with given routing strategy."""
        total_hits = 0
        total_segments = 0
        
        for i, path in enumerate(workload):
            if routing_strategy == "round_robin":
                worker_idx = self.round_robin_route(i)
            elif routing_strategy == "hash":
                worker_idx = self.hash_route(path)
            elif routing_strategy == "best_match":
                worker_idx = self.best_match_route(path)
            
            hits = self.workers[worker_idx].insert(path)
            total_hits += hits
            total_segments += len(path)
        
        return total_hits / total_segments if total_segments > 0 else 0

random.seed(42)
workload = generate_workload(300, n_users=8, n_topics=3)

strategies = ["round_robin", "hash", "best_match"]
print("=== Distributed Cache Routing Comparison ===")
print(f"Workers: 4, Cache per worker: 16 blocks, Workload: {len(workload)} requests\n")

for strategy in strategies:
    dist_cache = DistributedCache(n_workers=4, cache_size_per_worker=16)
    hit_rate = dist_cache.process_workload(workload, routing_strategy=strategy)
    print(f"{strategy:15s}: cache hit rate = {hit_rate:.1%}")

## 7. Performance Analysis: Cache Hit Rate vs Throughput

### 7.1 Theoretical Model

Let's model the relationship between cache hit rate and system throughput.

**Definitions:**
- `h` = cache hit rate (0 to 1)
- `T_prefill` = time for full prefill (no cache)
- `T_decode` = time per decode step
- `L` = average sequence length
- `N` = average number of generated tokens

**With cache hit rate `h`:**
- Effective prefill tokens = `L * (1 - h)`
- Prefill time = `T_prefill * (1 - h)` (linear approximation)
- Total time = `T_prefill * (1 - h) + N * T_decode`

**Throughput improvement:**
```
Throughput_ratio = Time_no_cache / Time_with_cache
                 = (T_prefill + N * T_decode) / (T_prefill * (1-h) + N * T_decode)
```

In [ ]:
# Performance model: cache hit rate vs throughput
import numpy as np

try:
    import matplotlib.pyplot as plt
    HAS_PLT = True
except ImportError:
    HAS_PLT = False

# Parameters
T_prefill = 10.0   # ms for full prefill
T_decode = 0.5     # ms per decode token
N_decode = 100     # average output tokens

hit_rates = np.linspace(0, 0.99, 100)
time_no_cache = T_prefill + N_decode * T_decode  # baseline

# Calculate throughput ratio for different hit rates
throughput_ratios = []
latency_reductions = []

for h in hit_rates:
    time_with_cache = T_prefill * (1 - h) + N_decode * T_decode
    throughput_ratios.append(time_no_cache / time_with_cache)
    latency_reductions.append((1 - time_with_cache / time_no_cache) * 100)

if HAS_PLT:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.plot(hit_rates * 100, throughput_ratios, 'b-', linewidth=2)
    ax1.set_xlabel('Cache Hit Rate (%)')
    ax1.set_ylabel('Throughput Improvement (x)')
    ax1.set_title('Throughput vs Cache Hit Rate')
    ax1.grid(True, alpha=0.3)
    ax1.axhline(y=1.0, color='r', linestyle='--', alpha=0.5, label='Baseline (no cache)')
    ax1.legend()
    
    ax2.plot(hit_rates * 100, latency_reductions, 'g-', linewidth=2)
    ax2.set_xlabel('Cache Hit Rate (%)')
    ax2.set_ylabel('Latency Reduction (%)')
    ax2.set_title('Latency Reduction vs Cache Hit Rate')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    # Text output
    print("Cache Hit Rate -> Throughput Improvement")
    for h, t in zip([0, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99], 
                     [throughput_ratios[0], throughput_ratios[25], throughput_ratios[50],
                      throughput_ratios[75], throughput_ratios[90], throughput_ratios[95],
                      throughput_ratios[99]]):
        print(f"  {h*100:5.0f}% -> {t:.2f}x")

In [ ]:
# Sensitivity analysis: impact of prefill-to-decode ratio

scenarios = {
    "Short output (N=10)": {"T_prefill": 10.0, "N_decode": 10},
    "Medium output (N=100)": {"T_prefill": 10.0, "N_decode": 100},
    "Long output (N=500)": {"T_prefill": 10.0, "N_decode": 500},
    "Long prompt, short output": {"T_prefill": 50.0, "N_decode": 20},
}

print("=== Throughput Improvement at Different Cache Hit Rates ===")
print(f"{'Scenario':<30s} | {'0%':>6s} | {'50%':>6s} | {'90%':>6s} | {'99%':>6s}")
print("-" * 70)

for name, params in scenarios.items():
    improvements = []
    for h in [0.0, 0.5, 0.9, 0.99]:
        time_base = params["T_prefill"] + params["N_decode"] * T_decode
        time_cached = params["T_prefill"] * (1 - h) + params["N_decode"] * T_decode
        improvements.append(time_base / time_cached)
    
    print(f"{name:<30s} | {improvements[0]:5.2f}x | {improvements[1]:5.2f}x | {improvements[2]:5.2f}x | {improvements[3]:5.2f}x")

print("\nKey insight: Cache benefits are MOST impactful when:")
print("  1. Prefill time dominates (long prompts)")
print("  2. Output is short (decode time is small)")
print("  3. This is exactly the chatbot scenario with system prompts!")

## 8. Deep Dive: Source Code Analysis

### 8.1 Key Files

| File | Purpose |
|------|--------|
| `sglang/srt/mem_cache/radix_cache.py` | Radix tree implementation, eviction logic |
| `sglang/srt/mem_cache/base_prefix_cache.py` | Base class for prefix cache |
| `sglang/srt/managers/scheduler.py` | Scheduling logic, cache-aware batch selection |
| `sglang/srt/mem_cache/chunk_cache.py` | Chunked cache implementation |

### 8.2 TreeNode Structure

```python
class TreeNode:
    """
    A node in the radix tree. Each node stores:
    - key: the token sequence for this edge
    - value: KV-cache block indices in the token pool
    - children: dict mapping next-token to child TreeNode
    - parent: reference to parent node
    - ref_counter: number of active requests using this node
    - last_access_time: for LRU eviction
    """
    
    def __init__(self):
        self.children = {}        # Dict[int, TreeNode]
        self.parent = None        # Optional[TreeNode]
        self.key = None           # Optional[List[int]] - token IDs
        self.value = None         # Optional[List[int]] - KV block indices
        self.lock_ref = 0         # Active references (cannot evict if > 0)
        self.last_access_time = 0 # For LRU ordering
```

### 8.3 Match Prefix Operation

```python
def match_prefix(self, key: List[int]) -> Tuple[List[int], TreeNode]:
    """
    Find the longest prefix of `key` that exists in the tree.
    
    Returns:
        - value: concatenated KV-cache block indices for the matched prefix
        - last_node: the deepest node in the match
    """
    node = self.root_node
    value = []
    
    i = 0
    while i < len(key):
        if key[i] in node.children:
            child = node.children[key[i]]
            child_key = child.key
            
            # Check if the child's key fully matches
            prefix_len = _common_prefix_length(key[i:], child_key)
            if prefix_len == len(child_key):
                # Full match: continue down the tree
                value.extend(child.value)
                node = child
                i += prefix_len
            else:
                # Partial match: return what we have
                break
        else:
            break
    
    return value, node
```

This operation is O(L) where L is the length of the input key, since each token is examined at most once as we traverse the tree.

In [ ]:
# Full radix tree implementation with all operations

class FullRadixTree:
    """Complete radix tree with match, insert, and evict operations."""
    
    def __init__(self, block_size=16):
        self.root = {"children": {}, "blocks": [], "ref_count": 0, "access_time": 0, "depth": 0}
        self.block_size = block_size
        self.time = 0
        self.next_block_id = 0
    
    def _alloc_blocks(self, n_tokens):
        """Allocate block IDs for n_tokens."""
        n_blocks = (n_tokens + self.block_size - 1) // self.block_size
        blocks = list(range(self.next_block_id, self.next_block_id + n_blocks))
        self.next_block_id += n_blocks
        return blocks
    
    def match_prefix(self, tokens):
        """
        Find longest matching prefix in the tree.
        Returns (matched_length, matched_blocks, last_node).
        """
        self.time += 1
        node = self.root
        matched_len = 0
        matched_blocks = []
        
        i = 0
        while i < len(tokens):
            token = tokens[i]
            if token in node["children"]:
                child = node["children"][token]
                child_tokens = child["tokens"]
                
                # Check how many tokens match
                j = 0
                while j < len(child_tokens) and i + j < len(tokens):
                    if tokens[i + j] != child_tokens[j]:
                        break
                    j += 1
                
                if j == len(child_tokens):
                    # Full match
                    matched_len += j
                    matched_blocks.extend(child["blocks"])
                    child["access_time"] = self.time
                    node = child
                    i += j
                else:
                    break  # Partial match
            else:
                break  # No match
        
        return matched_len, matched_blocks, node
    
    def insert(self, tokens):
        """Insert a sequence into the tree. Returns (matched_len, total_len)."""
        matched_len, _, node = self.match_prefix(tokens)
        
        # Insert remaining tokens
        remaining = tokens[matched_len:]
        if remaining:
            blocks = self._alloc_blocks(len(remaining))
            child = {
                "tokens": list(remaining),
                "blocks": blocks,
                "children": {},
                "ref_count": 0,
                "access_time": self.time,
                "depth": node["depth"] + 1,
                "parent": node
            }
            node["children"][remaining[0]] = child
        
        return matched_len, len(tokens)

# Demo: trace through match_prefix
tree = FullRadixTree(block_size=4)

# Insert some sequences
sequences = [
    [1, 2, 3, 4, 5, 6],        # Full sequence A
    [1, 2, 3, 4, 7, 8],        # Shares prefix [1,2,3,4] with A
    [1, 2, 3, 4, 5, 6, 9, 10], # Extension of A
    [1, 2, 3, 4, 5, 6],        # Exact repeat of A (full cache hit)
]

print("=== Radix Tree Match Prefix Demo ===")
for seq in sequences:
    matched, total = tree.insert(seq)
    print(f"Sequence {seq}: matched {matched}/{total} tokens (hit rate: {matched/total:.0%})")

## 9. Advanced Topic: Dynamic Cache Budget Allocation

### 9.1 The Budget Problem

GPU memory must be shared between:
1. **Model weights** (fixed)
2. **Active KV-cache** for running requests (dynamic)
3. **Cached KV-cache** for prefix reuse (dynamic)
4. **Temporary activations** (transient)

The challenge: how much memory to allocate for cached prefixes vs active generation?

```
Total GPU Memory: 80 GB
  - Model weights:      30 GB (fixed)
  - Activations:         5 GB (fixed)
  - Available for KV:   45 GB
    |- Active requests:  ?? GB
    |- Prefix cache:     ?? GB
```

### 9.2 SGLang's Approach

SGLang uses a **unified memory pool** where cached and active KV-cache share the same pool. The `evict()` function is called when the pool is full and new blocks are needed.

```python
# Simplified allocation flow
def allocate_for_request(self, request):
    needed_blocks = request.num_new_tokens // self.block_size
    
    while self.free_blocks < needed_blocks:
        # Evict cached prefixes to make room
        evicted = self.tree_cache.evict(self.block_size)
        if evicted == 0:
            # Cannot evict anything -- cache is all active
            return False  # Request must wait
        self.free_blocks += evicted
    
    self.free_blocks -= needed_blocks
    return True
```

This is elegant because:
- No explicit budget split needed
- Cache naturally grows when memory is available
- Cache shrinks automatically under pressure

In [ ]:
# Simulate dynamic cache budget under varying load

class UnifiedMemoryPool:
    """Simulates SGLang's unified memory pool for KV-cache."""
    
    def __init__(self, total_blocks):
        self.total_blocks = total_blocks
        self.active_blocks = 0      # Blocks used by running requests
        self.cached_blocks = 0      # Blocks used by prefix cache (evictable)
        self.history = []           # Track utilization over time
    
    @property
    def free_blocks(self):
        return self.total_blocks - self.active_blocks - self.cached_blocks
    
    def allocate(self, n_blocks, is_active=True):
        """Allocate blocks, evicting cache if needed."""
        while self.free_blocks < n_blocks and self.cached_blocks > 0:
            # Evict cached blocks
            evict_amount = min(n_blocks - self.free_blocks, self.cached_blocks)
            self.cached_blocks -= evict_amount
        
        if self.free_blocks >= n_blocks:
            if is_active:
                self.active_blocks += n_blocks
            else:
                self.cached_blocks += n_blocks
            return True
        return False
    
    def release_active(self, n_blocks, cache_fraction=0.5):
        """Release active blocks. Some become cached."""
        self.active_blocks -= n_blocks
        cache_blocks = int(n_blocks * cache_fraction)
        self.cached_blocks += cache_blocks
    
    def record_state(self):
        self.history.append({
            "active": self.active_blocks,
            "cached": self.cached_blocks,
            "free": self.free_blocks
        })

# Simulate a bursty workload
pool = UnifiedMemoryPool(total_blocks=100)

print("=== Unified Memory Pool Simulation ===")
print(f"Total capacity: {pool.total_blocks} blocks\n")

# Phase 1: Low load, cache builds up
print("Phase 1: Low load (5 requests)")
for i in range(5):
    pool.allocate(8, is_active=True)   # 8 blocks per request
    pool.record_state()
for i in range(5):
    pool.release_active(8, cache_fraction=0.75)  # Most becomes cached
    pool.record_state()
print(f"  Active={pool.active_blocks}, Cached={pool.cached_blocks}, Free={pool.free_blocks}")

# Phase 2: High load burst, cache gets evicted
print("\nPhase 2: High load burst (15 requests)")
for i in range(15):
    pool.allocate(6, is_active=True)
    pool.record_state()
print(f"  Active={pool.active_blocks}, Cached={pool.cached_blocks}, Free={pool.free_blocks}")

# Phase 3: Load decreases, cache rebuilds
print("\nPhase 3: Load decreases")
for i in range(15):
    pool.release_active(6, cache_fraction=0.5)
    pool.record_state()
print(f"  Active={pool.active_blocks}, Cached={pool.cached_blocks}, Free={pool.free_blocks}")

In [ ]:
# Visualize memory utilization over time
if HAS_PLT:
    history = pool.history
    steps = range(len(history))
    active = [h["active"] for h in history]
    cached = [h["cached"] for h in history]
    free = [h["free"] for h in history]
    
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.stackplot(steps, active, cached, free,
                 labels=['Active KV-cache', 'Prefix Cache', 'Free'],
                 colors=['#e74c3c', '#3498db', '#2ecc71'],
                 alpha=0.8)
    ax.set_xlabel('Time Step')
    ax.set_ylabel('Blocks')
    ax.set_title('Unified Memory Pool Utilization Over Time')
    ax.legend(loc='upper right')
    ax.set_ylim(0, pool.total_blocks)
    
    # Add phase annotations
    ax.axvline(x=10, color='black', linestyle='--', alpha=0.3)
    ax.axvline(x=25, color='black', linestyle='--', alpha=0.3)
    ax.text(3, 95, 'Low Load', ha='center', fontsize=10)
    ax.text(17, 95, 'High Load Burst', ha='center', fontsize=10)
    ax.text(33, 95, 'Load Decreasing', ha='center', fontsize=10)
    
    plt.tight_layout()
    plt.show()
else:
    print("Memory utilization over time (text summary):")
    for i, h in enumerate(pool.history[::5]):
        print(f"  Step {i*5:3d}: Active={h['active']:3d}, Cached={h['cached']:3d}, Free={h['free']:3d}")

## 10. Summary

### Key Takeaways

1. **Cache-aware scheduling** (LPM policy) significantly improves throughput by prioritizing requests with existing cache hits, reducing redundant prefill computation.

2. **LRU eviction** is the default and works well for most workloads. Frequency-based and hybrid policies can offer improvements for specific access patterns.

3. **Cache warming** eliminates cold-start penalties by pre-populating system prompts and common prefixes.

4. **Multi-level caching** (GPU -> CPU -> disk) extends effective cache size beyond GPU memory, since even disk access is faster than recomputation.

5. **Distributed cache coherence** requires cache-aware routing to ensure requests hit the right worker's cache.

6. **Cache benefits are maximized** when prefill time dominates total request time (long prompts, short outputs).

7. SGLang's **unified memory pool** elegantly handles the active-vs-cached trade-off by dynamically evicting cache under memory pressure.

## Exercises

### Exercise 1: Implement a Custom Eviction Policy

Implement a **W-TinyLFU** inspired eviction policy that:
1. Maintains a frequency sketch (approximate frequency count)
2. Uses a small window for recent entries
3. Compares candidates using estimated frequency

### Exercise 2: Cache Efficiency Analysis

Given a real-world access pattern (chatbot with 10 system prompts, 1000 users):
1. Measure cache hit rate as a function of cache size
2. Find the minimum cache size that achieves 90% hit rate
3. Plot the diminishing returns curve

### Exercise 3: Distributed Cache Routing

Implement a **consistent hashing** based router that:
1. Maps prefixes to workers using a hash ring
2. Handles worker failures by remapping to neighbors
3. Supports dynamic worker addition/removal

### Exercise 4: Implement Multi-Level Cache with Async Prefetch

Extend the multi-level cache to support **async prefetch**:
1. When a CPU cache hit occurs, asynchronously fetch related entries from disk
2. When an access pattern is detected, predictively promote entries
3. Measure the latency improvement from prefetching

In [ ]:
# Exercise 1 Starter Code: W-TinyLFU Eviction

class CountMinSketch:
    """Approximate frequency counter using Count-Min Sketch."""
    
    def __init__(self, width=64, depth=4):
        self.width = width
        self.depth = depth
        self.table = [[0] * width for _ in range(depth)]
    
    def _hash(self, key, seed):
        """Simple hash function."""
        return hash((key, seed)) % self.width
    
    def increment(self, key):
        for i in range(self.depth):
            self.table[i][self._hash(key, i)] += 1
    
    def estimate(self, key):
        return min(self.table[i][self._hash(key, i)] for i in range(self.depth))


class TinyLFUEviction(EvictionPolicy):
    """
    TODO: Implement W-TinyLFU eviction policy.
    
    Hints:
    1. Use CountMinSketch to track approximate frequencies
    2. When evicting, compare the frequency of the candidate victim
       with the frequency of the incoming item
    3. Only evict if the victim's frequency is lower
    """
    
    def __init__(self):
        self.sketch = CountMinSketch()
    
    def record_access(self, key):
        self.sketch.increment(key)
    
    def select_victim(self, leaves):
        # YOUR CODE HERE
        # Select the leaf with the lowest estimated frequency
        pass

print("Exercise starter code loaded. Implement the select_victim method!")

In [ ]:
# Exercise 2 Starter Code: Cache Size vs Hit Rate Analysis

def analyze_cache_size_impact(workload, cache_sizes):
    """
    TODO: For each cache size, measure the hit rate.
    
    Args:
        workload: list of request paths
        cache_sizes: list of cache sizes to test
    
    Returns:
        dict mapping cache_size -> hit_rate
    """
    results = {}
    
    for size in cache_sizes:
        # YOUR CODE HERE
        # Create a cache with the given size
        # Process the workload
        # Record the hit rate
        pass
    
    return results

# Generate realistic workload
# random.seed(42)
# workload = generate_workload(1000, n_users=100, n_topics=10)
# sizes = [8, 16, 32, 64, 128, 256, 512]
# results = analyze_cache_size_impact(workload, sizes)

print("Exercise 2 starter code loaded.")
print("Implement analyze_cache_size_impact and find the 90% hit rate threshold!")